In [1]:
"""
카카오프렌즈 스토어 카테고리 페이지에서 상품 링크(카테고리별 상위 10개)를 추출하고,
곧바로 각 상품 페이지의 리뷰까지 이어서 크롤링하는 단일 스파이더.

Twisted reactor는 커널(프로세스)당 1회만 시작할 수 있어서, "링크 수집"과
"리뷰 수집"을 서로 다른 CrawlerProcess로 나누면 두 번째 process.start()에서
ReactorNotRestartable 에러가 난다. 이를 피하기 위해 콜백을 체이닝해서
(parse_category -> parse_review) 하나의 스파이더, 하나의 process.start()로 처리한다.
"""

import scrapy
from scrapy_playwright.page import PageMethod


class KakaoAllInOneSpider(scrapy.Spider):
    name = "kakao_all_in_one"

    custom_settings = {
        "DOWNLOAD_HANDLERS": {
            "http": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
            "https": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
        },
        "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
        "PLAYWRIGHT_BROWSER_TYPE": "chromium",
        "PLAYWRIGHT_LAUNCH_OPTIONS": {"headless": True},
        "CONCURRENT_REQUESTS": 2,
        "DOWNLOAD_DELAY": 1.0,
        "ROBOTSTXT_OBEY": False,
        "USER_AGENT": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
    }

    CATEGORY_URLS = [
        "https://store.kakao.com/kakaofriends/category/7478375a1c7c4944befd972314b9e87c?sort=REVIEW_COUNT",
        "https://store.kakao.com/kakaofriends/category/69d5b59721fb456b8f1b1060144ae949?sort=REVIEW_COUNT",
    ]

    #LINKS_PER_CATEGORY = 10
    LINKS_PER_CATEGORY = 1
    MAX_MORE_CLICKS = 200

    async def start(self):
        for url in self.CATEGORY_URLS:
            yield scrapy.Request(
                url,
                meta=dict(
                    playwright=True,
                    playwright_include_page=True,
                    playwright_page_methods=[
                        # 상품 썸네일 링크가 렌더링될 때까지 대기 (SPA라서 필요)
                        PageMethod("wait_for_selector", "a.link_thumb", timeout=15000),
                    ],
                ),
                callback=self.parse_category,
                errback=self.errback_close_page,
            )

    async def parse_category(self, response):
        """1단계: 카테고리 페이지에서 상품 링크를 추출해 리뷰 파싱으로 이어간다."""
        page = response.meta["playwright_page"]
        await page.close()

        # 스크린샷에 나온 구조 기준: <a class="link_thumb" href="/kakaofriends/products/...">
        hrefs = response.css("a.link_thumb::attr(href)").getall()

        # 중복 제거 후 상위 10개만 유지, 절대 URL로 변환
        seen = []
        for href in hrefs:
            full_url = response.urljoin(href)
            if full_url not in seen:
                seen.append(full_url)
            if len(seen) >= self.LINKS_PER_CATEGORY:
                break

        self.logger.info("[%s] 상품 링크 %d개 추출", response.url, len(seen))

        for product_url in seen:
            yield scrapy.Request(
                product_url,
                meta={
                    "playwright": True,
                    "playwright_include_page": True,
                    "playwright_page_methods": [],
                },
                callback=self.parse_review,
                errback=self.errback_close_page,
                dont_filter=True,
            )

    async def parse_review(self, response):
        """2단계: 상품 페이지에서 리뷰 탭을 열고 '더보기'를 눌러가며 리뷰를 수집한다."""
        page = response.meta["playwright_page"]

        review_tab_selectors = [
            "a[data-tiara-layer='review']",
            "[data-tiara-action-name='리뷰 탭 클릭']",
            "a.link_tab:has-text('리뷰')",
            "text=리뷰",
            "a:has-text('리뷰')",
            "button:has-text('리뷰')",
        ]
        more_btn_selector = "a[data-tiara-layer='btn_review_list_more']"

        viewport_height = await page.evaluate("window.innerHeight")
        step = max(int(viewport_height * 0.6), 300)

        tab_clicked = False
        for i in range(15):
            for sel in review_tab_selectors:
                try:
                    el = await page.query_selector(sel)
                    if el:
                        await el.scroll_into_view_if_needed()
                        await el.click()
                        tab_clicked = True
                        self.logger.info("리뷰 탭 클릭 성공(스크롤 %d회차): %s", i + 1, sel)
                        await page.wait_for_timeout(2000)
                        break
                except Exception as e:
                    self.logger.debug("리뷰 탭 클릭 시도 실패(%s): %s", sel, e)
            if tab_clicked:
                break
            await page.evaluate(f"window.scrollBy(0, {step})")
            await page.wait_for_timeout(800)

        if not tab_clicked:
            self.logger.warning("리뷰 탭을 찾지 못했습니다: %s", response.url)

        click_count = 0
        while click_count < self.MAX_MORE_CLICKS:
            more_btn = await page.query_selector(more_btn_selector)
            if not more_btn:
                self.logger.info("'더보기' 버튼 없음 - 로딩 완료 (클릭 %d회)", click_count)
                break
            try:
                await more_btn.scroll_into_view_if_needed()
                await more_btn.click()
                click_count += 1
                await page.wait_for_timeout(1000)
                current_count = len(await page.query_selector_all("p.txt_review"))
                self.logger.info("'더보기' 클릭 %d회차 - 누적 리뷰 수: %d", click_count, current_count)
            except Exception as e:
                self.logger.warning("'더보기' 클릭 중 오류, 중단: %s", e)
                break

        product_name = await page.evaluate(
            """
            () => {
                const nameEl = document.querySelector('.txt_name');
                if (!nameEl) return null;
                const clone = nameEl.cloneNode(true);
                const screenOut = clone.querySelector('.screen_out');
                if (screenOut) screenOut.remove();
                return clone.textContent.trim();
            }
            """
        )
        self.logger.info("상품명: %s", product_name)

        reviews_data = await page.evaluate(
            """
            () => {
                const items = document.querySelectorAll('li.box_review, li.item-container.box_review');
                const results = [];
                items.forEach(item => {
                    const textEl = item.querySelector('p.txt_review');
                    if (!textEl) return;
                    const text = textEl.textContent.trim();
                    if (!text) return;

                    let rating = null;
                    const scoreEl = item.querySelector('.area_score em.img_shop');
                    if (scoreEl) {
                        const m = scoreEl.textContent.match(/(\\d+)/);
                        if (m) rating = parseInt(m[1], 10);
                    }

                    let date = null;
                    const infoItems = item.querySelectorAll('.list_reviewinfo li');
                    infoItems.forEach(li => {
                        const label = li.querySelector('strong.screen_out');
                        if (label && label.textContent.trim() === '작성일') {
                            const span = li.querySelector('span.txt_reviewinfo');
                            if (span) date = span.textContent.trim();
                        }
                    });

                    results.push({ text, rating, date });
                });
                return results;
            }
            """
        )
        self.logger.info("최종 추출된 리뷰 개수: %d", len(reviews_data))

        for idx, item in enumerate(reviews_data):
            yield {
                "product_url": response.url,
                "product_name": product_name,
                "review_index": idx,
                "review_text": item.get("text"),
                "rating": item.get("rating"),
                "date": item.get("date"),
            }

        await page.close()

    async def errback_close_page(self, failure):
        page = failure.request.meta.get("playwright_page")
        if page:
            await page.close()
        self.logger.error(repr(failure))

실행부

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()  # Jupyter의 asyncio 루프와 Twisted reactor 충돌 방지

# from scrapy.crawler import CrawlerProcess

# process = CrawlerProcess(settings={
#     **KakaoAllInOneSpider.custom_settings,
#     "FEEDS": {"reviews.jsonl": {"format": "jsonlines", "encoding": "utf8"}},
# })
# process.crawl(KakaoAllInOneSpider)
# # 링크 수집과 리뷰 수집이 하나의 스파이더로 합쳐졌으므로 process.start()는 한 번만 호출된다.
# process.start()

import asyncio
from scrapy.utils.reactor import install_reactor
install_reactor("twisted.internet.asyncioreactor.AsyncioSelectorReactor")

from scrapy.crawler import CrawlerRunner
from scrapy.utils.log import configure_logging
configure_logging()

runner = CrawlerRunner(settings={
    **KakaoAllInOneSpider.custom_settings,
    "FEEDS": {"reviews.jsonl": {"format": "jsonlines", "encoding": "utf8"}},
})

d = runner.crawl(KakaoAllInOneSpider)
await d.asFuture(asyncio.get_event_loop())   # 셀에서 바로 await

2026-07-08 08:11:12 [scrapy.addons] INFO: Enabled addons:
[]
2026-07-08 08:11:12 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-07-08 08:11:12 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-07-08 08:11:12 [scrapy.extensions.telnet] INFO: Telnet Password: 141982f378ae9171
2026-07-08 08:11:12 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-07-08 08:11:12 [scrapy.crawler] INFO: Overridden settings:
{'CONCURRENT_REQUESTS': 2,
 'DOWNLOAD_DELAY': 1.0,
 'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
               '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}
2026-07-08 08:11:12 [scrapy.middleware

## Transform

In [ ]:
"""
카카오스토어 베스트 상품 리뷰 ETL - Transform 단계
reviews.jsonl 을 읽어 캐릭터별 · 긍정/부정별 키워드 빈도를 집계해 저장한다.
"""

import json
import re
from collections import Counter

INPUT_PATH = "../../../../data/sentiment140/kakao_goods/reviews.jsonl"
OUTPUT_PATH = "../../../../data/sentiment140/kakao_goods/keywords_by_character.json"
TOP_N = 50

# 캐릭터는 별도 컬럼이 없으므로 상품명(product_name)에 이름이 포함되는지로 판별한다.
# "꿀잠친구_베어라이언" 같은 파생 상품명도 "라이언" 부분 문자열로 매칭되어
# 캐릭터는 항상 이 배열에 있는 정식 이름으로만 집계된다.
# 프로토타입 단계라 우선 춘식이/라이언만 수집한다.
CHARACTERS = ["춘식이", "라이언"]

POSITIVE_MIN_RATING = 3

NEGATIVE_MAX_RATING = 2

STOPWORDS = {
    "너무", "정말", "진짜", "그냥", "완전", "아주", "이제", "많이",
    "제품", "상품", "구매", "배송", "사용",
    "이거", "저거", "그거", "이것", "저것", "그것",
    "합니다", "했어요", "해요", "있어요", "없어요", "같아요", "같은",
    "위해", "통해", "대해", "그리고", "그런데", "하지만", "그래서", "때문에",
    "역시", "완전히", "정도",
}

TOKEN_PATTERN = re.compile(r"[가-힣]{2,}")


def load_reviews(path):
    reviews = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            reviews.append(json.loads(line))
    return reviews


def match_characters(product_name):
    """하드코딩된 CHARACTERS 배열을 상품명에서 부분 문자열로 매칭한다."""
    return [name for name in CHARACTERS if name in product_name]


def classify_sentiment(rating):
    if not isinstance(rating, int):
        return None
    if rating >= POSITIVE_MIN_RATING:
        return "positive"
    if rating <= NEGATIVE_MAX_RATING:
        return "negative"
    return None  # 중립 평점(3점)은 긍/부정 집계에서 제외


def tokenize(text, exclude_words):
    """한글 2음절 이상 토큰만 추출하고 불용어·캐릭터명을 제거한다."""
    tokens = TOKEN_PATTERN.findall(text)
    return [t for t in tokens if t not in STOPWORDS and t not in exclude_words]


def extract_concept_keywords(product_name, exclude_words):
    """상품명에서 매칭된 캐릭터 이름을 제거한 나머지를 '컨셉 키워드'로 추출한다."""
    text = product_name
    for name in exclude_words:
        text = text.replace(name, " ")
    tokens = TOKEN_PATTERN.findall(text)
    return [t for t in tokens if t not in STOPWORDS and t not in exclude_words]


def build_keyword_counters(reviews):
    counters = {name: {"positive": Counter(), "negative": Counter()} for name in CHARACTERS}

    for review in reviews:
        sentiment = classify_sentiment(review.get("rating"))
        if sentiment is None:
            continue

        product_name = review.get("product_name", "")
        matched = match_characters(product_name)
        if not matched:
            continue

        exclude = set(matched)
        review_tokens = set(tokenize(review.get("review_text", ""), exclude_words=exclude))
        concept_tokens = set(extract_concept_keywords(product_name, exclude_words=exclude))  # 추가

        tokens = review_tokens | concept_tokens  # 리뷰 본문 키워드 + 상품명 컨셉 키워드 합침
        for character in matched:
            counters[character][sentiment].update(tokens)

    return counters


def to_serializable(counters, top_n=TOP_N):
    return {
        character: {
            sentiment: counter.most_common(top_n)
            for sentiment, counter in sentiments.items()
        }
        for character, sentiments in counters.items()
    }


def save_keywords(counters, path, top_n=TOP_N):
    data = to_serializable(counters, top_n=top_n)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"캐릭터별 긍정/부정 키워드 저장 완료 -> {path}")


def main():
    reviews = load_reviews(INPUT_PATH)
    counters = build_keyword_counters(reviews)
    save_keywords(counters, OUTPUT_PATH)

    for character, sentiments in counters.items():
        pos_total = sum(sentiments["positive"].values())
        neg_total = sum(sentiments["negative"].values())
        if pos_total or neg_total:
            print(f"{character}: positive={pos_total}개, negative={neg_total}개")


if __name__ == "__main__":
    main()


## Load

In [ ]:
!pip install wordcloud -q
!apt-get -qq install fonts-nanum > /dev/null

import json
import os
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from wordcloud import WordCloud

FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

def register_korean_font(font_path: str) -> str:
    fm.fontManager.addfont(font_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
    plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False
    return font_name

def build_wordcloud(word_freq_pairs, font_path, colormap):
    freq_dict = {word: count for word, count in word_freq_pairs}
    wc = WordCloud(
        font_path=font_path,
        width=800,
        height=600,
        background_color="white",
        colormap=colormap,
        prefer_horizontal=0.9,
        max_words=100,
    )
    wc.generate_from_frequencies(freq_dict)
    return wc

def make_character_figure(character, data, font_path, outdir="outputs"):
    os.makedirs(outdir, exist_ok=True)
    positive = data.get("positive", [])
    negative = data.get("negative", [])

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    if positive:
        wc_pos = build_wordcloud(positive, font_path, colormap="Blues")
        axes[0].imshow(wc_pos, interpolation="bilinear")
    axes[0].set_title(f"{character} - 긍정 키워드", fontsize=16)
    axes[0].axis("off")

    if negative:
        wc_neg = build_wordcloud(negative, font_path, colormap="Reds")
        axes[1].imshow(wc_neg, interpolation="bilinear")
    axes[1].set_title(f"{character} - 부정 키워드", fontsize=16)
    axes[1].axis("off")

    fig.suptitle(f"'{character}' 리뷰 키워드 워드클라우드", fontsize=18, y=1.02)
    fig.tight_layout()

    safe_name = character.replace("/", "_")
    out_path = os.path.join(outdir, f"wordcloud_{safe_name}.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()  #

In [ ]:
register_korean_font(FONT_PATH)

json_path = "../../../../data/sentiment140/kakao_goods/keywords_by_character.json"
with open(json_path, encoding="utf-8") as f:
    data_by_character = json.load(f)

saved_paths = []
for character, data in data_by_character.items():
    out_path = make_character_figure(character, data, FONT_PATH)
    saved_paths.append(out_path)
    print(f"저장 완료: {out_path}")

saved_paths